# This notebook performs fault slip analysis (FSA) based on principal stresses from CMG geomechanical simulations. The following data are needed:
- Grid cell coordinates and fault ID extracted from Petrel exported files.
- Principal stresses (STRESMXP, STRESMNP, STRESINT) extracted from CMG simulation results (sr3/gmch.sr3 -> rwo -> numpy).

# Step 2: Estimate Mw exceedance probability

for fault0 only, following two steps:
1. evaluate if a fault would slip
2. if yes, then estimate estimated Mw exceedance proability

In [13]:
import numpy as np
from pathlib import Path
from scipy.stats import gaussian_kde

################## User Inputs ############################## 
# name_prefix = '251104'; n_cases = 86; n_faults = 12; n_times = 6; 
fault_id = 0
slip_threshold = 0.004 # for fault0 0.005 corresponds to at least 6 cells
n_samples = 100_000 # for Monte Carlo sampling
# fault_area = 16_985_344.51 # fault total area in m2, for fault0
Mw_target = 1.5 # target magnitude for exceedance probability
# set up path
# base_path = Path('.')
# count total number of fault cells using the whole grid (not just in the reservior)
# coor_fault = np.load('data/coor_fault/JD_Sula_2025_gmc_coor&fault_reservoir.npy')
coor_fault_all = np.load(base_path/'data/coor_fault/JD_Sula_2025_gmc_coor&fault.npy')
total_failed_cells = np.load(fault_slip_analysis_folder/f'{name_prefix}_total_failed_cells.npy')
parameters = pd.read_csv(base_path/'data'/'params_responses'/f'{name_prefix}_CMG_parameters.csv')
################## End of User Inputs #######################

# fault_cell_count = np.full((n_faults), np.nan)
fault_cell_count_all = np.full((n_faults), np.nan)
for fault_id_itr in range(0,n_faults):
    fault_id_mask_all = (coor_fault_all[:,:,:,3] == fault_id_itr)
    fault_cell_count_all[fault_id_itr] = np.count_nonzero(fault_id_mask_all)

failed_cell_ratio = total_failed_cells / fault_cell_count_all[np.newaxis,:,np.newaxis]

# estimate Mw through Monte Carlo sampling
# initialize an array to hold Mw exceedance probability for all cases and time steps
# Mw_exceedance = np.full((n_cases,n_times), np.nan) # array shape (n_cases,n_times)
Mw_exceedance = np.zeros((n_cases,n_times)) # array shape (n_cases,n_times)

mean_tau = np.load(fault_slip_analysis_folder/f'{name_prefix}_mean_tau.npy')

# run analysis
for case_num in tqdm(range(1,n_cases+1), desc='Estimating moment magnitude'):
    # extract the fault area from the parameter dataframe
    fault_area = parameters.loc[parameters["case_name"] == f'case{case_num}','A_m2'].iloc[0]

    for time_step in range(n_times):
        failure_ratio = failed_cell_ratio[case_num-1,fault_id,time_step]
        if failure_ratio >= slip_threshold:

            mean_shear_stress = mean_tau[case_num-1,fault_id,time_step]
            stress_drop = np.random.uniform(mean_shear_stress * 0.1, mean_shear_stress * 0.9, n_samples)
            rupture_area = np.random.uniform(fault_area * failure_ratio, fault_area, n_samples)
            # calculate moment magnitude
            Mw = (np.log10(stress_drop * rupture_area ** 1.5) - 9.1) * 2 / 3

            # Kernel density for smooth PDF estimate
            kde = gaussian_kde(Mw)
            Mw_grid = np.linspace(Mw.min(), Mw.max(), 1000)
            pdf = kde(Mw_grid)

            # Empirical CDF
            Mw_sorted = np.sort(Mw)
            cdf = np.arange(1, len(Mw_sorted) + 1) / len(Mw_sorted)

            # Exceedance probability = 1 - CDF
            exceedance_prob = 1 - cdf

            # Find exceedance probability at target Mw
            Mw_exceedance[case_num-1,time_step] = np.mean(Mw > Mw_target)

# create headers
# Define column names in a list
column_headers = [f'time{i}' for i in range(n_times)]
# Join the list of names into a single string using your delimiter
header_string = ','.join(column_headers)
np.savetxt(fault_slip_analysis_folder/f'{name_prefix}_Mw_exceedance{Mw_target}_fault0.csv',Mw_exceedance,delimiter=',',fmt='%.4f',header=header_string,comments='')

# print(Mw_exceedance)

Estimating moment magnitude: 100%|██████████| 93/93 [00:31<00:00,  2.95it/s]


In [14]:
Mw_exceedance
print(np.nonzero(Mw_exceedance[:,2]))
print(np.count_nonzero(Mw_exceedance[:,2]))
Mw_exceedance[:,2]
# np.mean(Mw_exceedance[:,2])

(array([17, 18, 20, 25, 26, 32, 40, 46, 57, 62, 67, 70, 72, 75, 83, 85, 87,
       88]),)
18


array([0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.2414 , 0.50831, 0.     , 0.37784,
       0.     , 0.     , 0.     , 0.     , 0.36736, 0.50896, 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.26962, 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.21335, 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.53321, 0.     , 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
       0.     , 0.41404, 0.     , 0.     , 0.     , 0.     , 0.55803,
       0.     , 0.     , 0.     , 0.     , 0.26418, 0.     , 0.     ,
       0.58314, 0.     , 0.41158, 0.     , 0.     , 0.52142, 0.     ,
       0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.56218,
       0.     , 0.44224, 0.     , 0.49464, 0.41865, 0.     , 0.     ,
       0.     , 0.     ])

In [118]:
parameters = pd.read_csv(base_path/'data'/'params_responses'/f'{name_prefix}_CMG_parameters.csv')
parameters['SH_azi_deg'].iloc[np.nonzero(Mw_exceedance[:,2])]

17    319.8449
25    319.9424
27    319.6814
30    318.7329
53    319.9319
68    318.4956
82    319.8959
88    319.6633
Name: SH_azi_deg, dtype: float64

## calculate the (number of cases with fault slip / total cases)

In [50]:
import numpy as np
from pathlib import Path

name_prefix = '250922'
base_path = Path('.')

FSA_combined = np.load(base_path/'data'/f'{name_prefix}_FSA_combined.npy')
no_slip_count = np.sum(FSA_combined == 0, axis=0)
slip_probability = 1 - no_slip_count/FSA_combined.shape[0]
# np.savetxt(base_path/'data'/f'{save_file_prefix}_FSA_probability.csv',slip_probability,delimiter=",",fmt="%.4f")
# Save as CSV
df = pd.DataFrame(
    slip_probability,
    columns=[f"time_{t}" for t in range(n_times)]
)
df.insert(0, "fault_id", range(n_faults))

df.to_csv(base_path/'data'/f'{name_prefix}_FSA_probability.csv', index=False, float_format="%.4f")
df

,fault_id,time_0,time_1,time_2,time_3,time_4,time_5
0,0,0.0,0.0,0.022222,0.033333,0.033333,0.022222
1,1,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,2,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,3,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,4,0.0,0.0,0.033333,0.011111,0.011111,0.011111
5,5,0.0,0.0,0.000000,0.000000,0.000000,0.000000
6,6,0.0,0.0,0.033333,0.066667,0.044444,0.044444
7,7,0.0,0.0,0.000000,0.000000,0.000000,0.000000
8,8,0.0,0.0,0.000000,0.000000,0.000000,0.000000
9,9,0.0,0.0,0.000000,0.000000,0.000000,0.000000


# calculate responses for DGSA sensitivity analysis

stress ratio and Mw exceedance probability for fault0

In [174]:
import numpy as np
from pathlib import Path

Mw_target = 1.5 # target magnitude for exceedance probability
name_prefix = '251023'; fault_id = [0]; year = 2050; year_list = [2030, 2040, 2050, 2060, 2550, 3050]
base_path = Path('.')
mean_stress_ratio = np.load(base_path/'data'/f'{name_prefix}_mean_stress_ratio.npy')
Mw_exceedance_prob = np.loadtxt(base_path/'data'/f'{name_prefix}_Mw_exceedance{Mw_target}_fault0.csv',delimiter=',',skiprows=1)
resp = mean_stress_ratio[:,fault_id,year_list.index(year)]
resp = np.insert(resp,1,Mw_exceedance_prob[:,year_list.index(year)],axis=1)

np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_fault{fault_id}.csv',resp,delimiter=',',fmt='%.4f',header=f'stress_ratio,exceedance{Mw_target}_prob',comments='')
print(mean_stress_ratio.shape)
print(Mw_exceedance_prob.shape)
resp

(90, 12, 6)
(90, 6)


array([[0.56060279, 0.        ],
       [0.55464004, 0.        ],
       [0.55763848, 0.        ],
       [0.55605423, 0.        ],
       [0.58277697, 0.        ],
       [0.61458712, 0.5636    ],
       [0.59395949, 0.2692    ],
       [0.55113539, 0.        ],
       [0.48577605, 0.        ],
       [0.57148702, 0.        ],
       [0.53368451, 0.        ],
       [0.58456919, 0.        ],
       [0.50165047, 0.        ],
       [0.54394929, 0.        ],
       [0.52532833, 0.        ],
       [0.51560108, 0.        ],
       [0.53859826, 0.        ],
       [0.60359795, 0.4531    ],
       [0.55071477, 0.        ],
       [0.52902567, 0.        ],
       [0.52619816, 0.        ],
       [0.53513798, 0.        ],
       [0.55192879, 0.        ],
       [0.59228606, 0.226     ],
       [0.54370425, 0.        ],
       [0.60983362, 0.4328    ],
       [0.48385002, 0.        ],
       [0.58667367, 0.        ],
       [0.60059314, 0.377     ],
       [0.54958008, 0.        ],
       [0.